# Lab 07 — RNN, LSTM and GRU for sequence prediction

**Dataset:** [huggingface/electricity-production](https://huggingface.co/datasets/huggingface/electricity-production), a compact monthly electricity-production time series stored as Parquet. We create chronological train/validation/test periods, fit preprocessing on training data only, and compare recurrent cells under the same window, optimizer and training budget. This short series is for teaching the mechanics, not a production forecasting benchmark.

## The math behind the experiment

A vanilla RNN carries a hidden state forward:

$$
h_t=\phi(W_xx_t+W_hh_{t-1}+b).
$$

The same $W_h$ is reused at every time step. During backpropagation through time, gradients repeatedly multiply by derivatives associated with $W_h$. If those products become very small, long-range information is hard to learn; if they become very large, gradients can explode.

An LSTM introduces gates to control information flow. A simplified cell update is

$$
c_t=f_t\odot c_{t-1}+i_t\odot\tilde{c}_t.
$$

In [ ]:
!pip -q install datasets torch pandas numpy matplotlib
from datasets import load_dataset
import numpy as np,pandas as pd,torch
from torch import nn
raw=load_dataset("huggingface/electricity-production",split="train")
raw_values=np.asarray(raw["IPG2211A2N"],dtype=np.float32)
raw_values=raw_values[np.isfinite(raw_values)][:2000]
cut_train=int(.70*len(raw_values)); cut_val=int(.85*len(raw_values))
# Fit normalization on the training period only; reuse it for later periods.
train_mean=float(raw_values[:cut_train].mean())
train_std=float(raw_values[:cut_train].std()+1e-6)
vals=torch.tensor((raw_values-train_mean)/train_std,dtype=torch.float32)
T=24

def make_windows(series,T):
 target_idx=np.arange(T,len(series))
 X=torch.stack([series[t-T:t] for t in target_idx])
 y=series[torch.as_tensor(target_idx,dtype=torch.long)]
 train_ids=np.where(target_idx<cut_train)[0]
 val_ids=np.where((target_idx>=cut_train)&(target_idx<cut_val))[0]
 test_ids=np.where(target_idx>=cut_val)[0]
 return (X[train_ids],y[train_ids],X[val_ids],y[val_ids],X[test_ids],y[test_ids])

Xtr,ytr,Xval,yval,Xte,yte=make_windows(vals,T)
print("Monthly electricity-production series:",tuple(vals.shape))
print("Window counts train/validation/test:",len(Xtr),len(Xval),len(Xte))
print("Scaler fit period ends at raw index",cut_train-1)

## Step 1 — Prediction

RNNs should learn short temporal structure but can struggle as dependency length grows. LSTM/GRU gates should make long-range credit assignment easier. The comparison must use the same hidden size and training budget.

In [ ]:
class RNNModel(nn.Module):
 def __init__(self,kind="RNN",h=32):
  super().__init__(); R={"RNN":nn.RNN,"LSTM":nn.LSTM,"GRU":nn.GRU}[kind]; self.r=R(1,h,batch_first=True); self.o=nn.Linear(h,1)
 def forward(self,x): return self.o(self.r(x.unsqueeze(-1))[0][:,-1]).squeeze(-1)

def fit(kind,T=24,epochs=6):
 Xtr2,ytr2,Xval2,yval2,_,_=make_windows(vals,T)
 m=RNNModel(kind); opt=torch.optim.Adam(m.parameters(),1e-3); loss=nn.MSELoss()
 for _ in range(epochs):
  opt.zero_grad(); l=loss(m(Xtr2),ytr2); l.backward(); opt.step()
 with torch.no_grad():
  train_mse=loss(m(Xtr2),ytr2).item(); val_mse=loss(m(Xval2),yval2).item()
 return train_mse,val_mse,sum(p.numel() for p in m.parameters())

rows=[]
for k in ["RNN","LSTM","GRU"]:
 rows.append([k,*fit(k,T=24)])
pd.DataFrame(rows,columns=["cell","train_MSE","validation_MSE","parameters"])

## Step 2 — Dependency-length intervention

Train on the past period and compare validation error for windows of 8, 24 and 48 steps; reserve the final chronological test period for one selected configuration. If performance degrades as the window grows, that is evidence about optimization/memory—not proof that the architecture “cannot remember.”

In [ ]:
def window_experiment(kind,T):
 Xtr2,ytr2,Xval2,yval2,_,_=make_windows(vals,T)
 m=RNNModel(kind); opt=torch.optim.Adam(m.parameters(),1e-3); loss=nn.MSELoss()
 for _ in range(5):
  opt.zero_grad(); l=loss(m(Xtr2),ytr2); l.backward(); opt.step()
 with torch.no_grad(): return loss(m(Xval2),yval2).item()
rows=[[T,window_experiment("RNN",T),window_experiment("LSTM",T),window_experiment("GRU",T)] for T in [8,24,48]]
pd.DataFrame(rows,columns=["window","RNN_validation_MSE","LSTM_validation_MSE","GRU_validation_MSE"])

## Visualize memory — performance vs sequence length

A longer input window gives a model access to more history, but it also changes optimization difficulty and the amount of information the model must use. Compare validation error across RNN, LSTM, and GRU cells rather than assuming a gated model always wins. The final test period is not used for this choice.

In [ ]:
import matplotlib.pyplot as plt

window_results = pd.DataFrame(rows, columns=["window", "RNN_test_MSE", "LSTM_test_MSE", "GRU_test_MSE"])
fig, ax = plt.subplots(figsize=(8, 4.8))
for col in ["RNN_validation_MSE", "LSTM_validation_MSE", "GRU_validation_MSE"]:
    ax.plot(window_results["window"], window_results[col], marker="o", label=col.replace("_validation_MSE", ""))
ax.set_title("Recurrent cell comparison by context length")
ax.set_xlabel("Input window length (time steps)")
ax.set_ylabel("Validation MSE")
ax.legend()
ax.grid(alpha=.25)
plt.tight_layout()
plt.show()

## Final chronological test evaluation

Select the cell type and context length with the lowest validation MSE. Refit that configuration on all windows whose targets fall before the validation/test boundary, then evaluate it on the final chronological test windows once. Normalization statistics remain those estimated from the training period, so no future information enters preprocessing.


In [ ]:
long_rows=[]
for T in [8,24,48]:
    for kind in ["RNN","LSTM","GRU"]:
        long_rows.append([kind,T,window_experiment(kind,T)])
validation_grid=pd.DataFrame(long_rows,columns=["cell","window","validation_MSE"])
best=validation_grid.loc[validation_grid["validation_MSE"].idxmin()]
best_kind=str(best["cell"]); best_T=int(best["window"])
Xtr2,ytr2,Xval2,yval2,Xte2,yte2=make_windows(vals,best_T)
Xfit=torch.cat([Xtr2,Xval2]); yfit=torch.cat([ytr2,yval2])
torch.manual_seed(202)
final_model=RNNModel(best_kind)
final_opt=torch.optim.Adam(final_model.parameters(),lr=1e-3); loss_fn=nn.MSELoss()
for _ in range(5):
    final_opt.zero_grad(); loss_fn(final_model(Xfit),yfit).backward(); final_opt.step()
with torch.no_grad(): final_test_mse=float(loss_fn(final_model(Xte2),yte2).item())
print({"validation_selected_cell":best_kind,"window":best_T,"final_test_MSE":final_test_mse,"train_mean":train_mean,"train_std":train_std})


## Conclusion / answer key

Expected: the gated models should often be more stable as context length grows, although exact ranking is data- and hyperparameter-dependent. Report what happened rather than asserting a universal LSTM/GRU superiority.

### Research extension
Repeat with chronological train/test evaluation and three seeds; plot error against dependency length.